In [6]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from sklearn.metrics import confusion_matrix, classification_report, cohen_kappa_score, accuracy_score
from PIL import Image
import timm

# --- CONFIG ---
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
BASE_PATH = '/kaggle/input/datasets/mariaherrerot/aptos2019/'
TRAIN_IMG_DIR = os.path.join(BASE_PATH, 'train_images/train_images')
VAL_IMG_DIR = os.path.join(BASE_PATH, 'val_images/val_images')
IMG_SIZE = 256
BATCH_SIZE = 32
LR = 6e-5 # Slightly lower for fine-grained attention tuning
EPOCHS = 20

# --- NOVELTY: DUAL-PATH MULTI-SCALE ATTENTION ---
class MultiScaleAttention(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.query_conv = nn.Conv2d(in_channels, in_channels // 8, 1)
        self.key_conv = nn.Conv2d(in_channels, in_channels // 8, 1)
        self.value_conv = nn.Conv2d(in_channels, in_channels, 1)
        self.gamma = nn.Parameter(torch.zeros(1))
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        batch, channels, height, width = x.size()
        proj_query = self.query_conv(x).view(batch, -1, width * height).permute(0, 2, 1)
        proj_key = self.key_conv(x).view(batch, -1, width * height)
        energy = torch.bmm(proj_query, proj_key)
        attention = self.softmax(energy)
        proj_value = self.value_conv(x).view(batch, -1, width * height)

        out = torch.bmm(proj_value, attention.permute(0, 2, 1))
        out = out.view(batch, channels, height, width)
        return self.gamma * out + x

# --- ARCHITECTURE ---
class AptosDPMSNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Using a backbone that allows intermediate feature access
        self.backbone = timm.create_model('tf_efficientnetv2_s', pretrained=True, num_classes=0, global_pool='')
        # Final stage has 1280 channels
        self.attention = MultiScaleAttention(1280)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1280, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 5)
        )

    def forward(self, x):
        features = self.backbone(x)
        attended_features = self.attention(features)
        pooled = self.pool(attended_features)
        return self.head(pooled)

# --- UTILITIES ---
class AptosDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(os.path.join(BASE_PATH, csv_file))
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, f"{self.df.iloc[idx, 0]}.png")
        image = Image.open(img_path).convert('RGB')
        label = self.df.iloc[idx, 1]
        if self.transform: image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

def run_training():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = AptosDPMSNet().to(device)

    # Balanced Sampler focusing on Severe (Class 3)
    train_df = pd.read_csv(os.path.join(BASE_PATH, 'train_1.csv'))
    counts = train_df.diagnosis.value_counts()
    class_weights = 1. / counts
    class_weights[3] *= 2.5 # Extra emphasis on missing severe cases
    sample_weights = class_weights[train_df.diagnosis].values
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

    train_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomAffine(degrees=15, translate=(0.1, 0.1)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    train_loader = DataLoader(AptosDataset('train_1.csv', TRAIN_IMG_DIR, train_tf), 
                              batch_size=BATCH_SIZE, sampler=sampler, num_workers=4)
    val_loader = DataLoader(AptosDataset('valid.csv', VAL_IMG_DIR, train_tf), batch_size=BATCH_SIZE)

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    best_kappa = 0
    for epoch in range(EPOCHS):
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()

        model.eval()
        v_preds, v_labels = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                v_preds.extend(torch.argmax(model(imgs), 1).cpu().numpy())
                v_labels.extend(labels.cpu().numpy())
        
        kappa = cohen_kappa_score(v_labels, v_preds, weights='quadratic')
        print(f"Epoch {epoch+1} | Kappa: {kappa:.4f} | Acc: {accuracy_score(v_labels, v_preds):.4f}")
        
        if kappa > best_kappa:
            best_kappa = kappa
            torch.save(model.state_dict(), 'dpms_aptos_model.pth')

    # FINAL DETAILED REPORT
    print("\n" + "="*60)
    print("             FINAL CLINICAL DOCUMENTATION")
    print("="*60)
    model.load_state_dict(torch.load('dpms_aptos_model.pth'))
    model.eval()
    
    f_preds, f_labels = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            f_preds.extend(torch.argmax(model(imgs), 1).cpu().numpy())
            f_labels.extend(labels.cpu().numpy())

    cm = confusion_matrix(f_labels, f_preds)
    classes = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
    
    print(f"Final Accuracy: {accuracy_score(f_labels, f_preds):.4%}")
    print(f"Final Kappa:    {cohen_kappa_score(f_labels, f_preds, weights='quadratic'):.4f}")
    print("-" * 60)
    
    for i in range(5):
        tp = cm[i, i]
        fn = sum(cm[i, :]) - tp
        fp = sum(cm[:, i]) - tp
        tn = sum(cm.flatten()) - (tp + fn + fp)
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        print(f"{classes[i]:<15} | Sensitivity: {sens:.4f} | Specificity: {spec:.4f}")

if __name__ == "__main__":
    run_training()

Epoch 1 | Kappa: 0.8351 | Acc: 0.6612
Epoch 2 | Kappa: 0.8126 | Acc: 0.7158
Epoch 3 | Kappa: 0.8375 | Acc: 0.7486
Epoch 4 | Kappa: 0.8511 | Acc: 0.7814
Epoch 5 | Kappa: 0.8552 | Acc: 0.8060
Epoch 6 | Kappa: 0.8755 | Acc: 0.8251
Epoch 7 | Kappa: 0.8923 | Acc: 0.8361
Epoch 8 | Kappa: 0.8772 | Acc: 0.7923
Epoch 9 | Kappa: 0.8776 | Acc: 0.7978
Epoch 10 | Kappa: 0.8792 | Acc: 0.7951
Epoch 11 | Kappa: 0.8699 | Acc: 0.7923
Epoch 12 | Kappa: 0.8703 | Acc: 0.8005
Epoch 13 | Kappa: 0.8702 | Acc: 0.8142
Epoch 14 | Kappa: 0.8764 | Acc: 0.8142
Epoch 15 | Kappa: 0.8910 | Acc: 0.8224
Epoch 16 | Kappa: 0.8776 | Acc: 0.8169
Epoch 17 | Kappa: 0.8808 | Acc: 0.8142
Epoch 18 | Kappa: 0.8812 | Acc: 0.7978
Epoch 19 | Kappa: 0.8811 | Acc: 0.8169
Epoch 20 | Kappa: 0.8619 | Acc: 0.7951

             FINAL CLINICAL DOCUMENTATION
Final Accuracy: 80.8743%
Final Kappa:    0.8743
------------------------------------------------------------
No DR           | Sensitivity: 0.9942 | Specificity: 0.9794
Mild            |